# 🚀 DINOv2-TVPR: Huấn Luyện & Web App Demo trên Google Colab

**Dự án:** Text-to-Video Person Retrieval (TVPR) & Continuous Tracking Thuần Tiếng Việt (100% Vietnamese)
- **Thị giác (Vision Backbone):** **Meta DINOv2** (`facebook/dinov2-small`, 384-D, 21M tham số) trích xuất đặc trưng trang phục người cực nhạy, kết hợp **Temporal Attention Aggregator** gom chuỗi khung hình thời gian.
- **Ngôn ngữ (Text Backbone):** **VinAI PhoBERT v2** (`vinai/phobert-base-v2`) hiểu sâu sắc ngữ cảnh câu mô tả tiếng Việt.
- **Không gian chung & Loss:** Common Space 256-D + **Symmetric Cross-Modal InfoNCE** (CLIP-style) với tham số nhiệt độ học linh hoạt `logit_scale` (chống sụp đổ biểu diễn 100%).

### 📑 Các Cell Trong Notebook:
1. **Cell 0:** Kiểm tra GPU & Cài đặt thư viện.
2. **Cell 1:** Mount Google Drive & Tùy chọn sao chép Dataset sang SSD Colab (`COPY_TO_LOCAL_SSD = True/False`).
3. **Cell 2:** Cấu hình huấn luyện tối ưu và chạy `train.py`, tự động lưu checkpoint về Google Drive.
4. **Cell 3:** Web App Gradio tương tác trực tiếp trong cell (tìm kiếm & tracking người từ video theo câu mô tả tiếng Việt, sinh link public `gradio.live`).


In [ ]:
# ==============================================================================
# 0. KIỂM TRA GPU & CÀI ĐẶT THƯ VIỆN PHỤ THUỘC
# ==============================================================================
import torch
import os

print(f"[*] PyTorch Version: {torch.__version__}")
print(f"[*] CUDA Khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[*] Thiết bị GPU: {torch.cuda.get_device_name(0)}")
    !nvidia-smi
else:
    print("[!] CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU hoặc A100.")

print("\n[*] Đang cài đặt các thư viện cần thiết...")
!pip install -q transformers pyyaml opencv-python tqdm pandas gradio ultralytics pyvi safetensors
print("[+] Hoàn tất cài đặt môi trường!")


## 📁 Cell 1: Mount Google Drive & Quản Lý Dữ Liệu

- **`DRIVE_PROJECT_ROOT`**: Thư mục dự án `PBL6` trên Google Drive của bạn.
- **`COPY_TO_LOCAL_SSD`**:
  - `True` *(Khuyên dùng)*: Tự động sao chép thư mục `dataset/` (hoặc giải nén `dataset.zip`) sang SSD cục bộ của Colab (`/content/dataset`). Tốc độ nạp video nhanh gấp **5 - 10 lần**, tránh nghẽn I/O qua Google Drive.
  - `False`: Đọc video trực tiếp từ Google Drive.


In [ ]:
# ==============================================================================
# CELL 1: MOUNT GOOGLE DRIVE & TÙY CHỌN COPY DATASET SANG SSD COLAB
# ==============================================================================
import os
import shutil
import zipfile
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# --- CẤU HÌNH ĐƯỜNG DẪN DỰ ÁN TRÊN GOOGLE DRIVE ---
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/PBL6"

# CỜ TÙY CHỌN: Copy dataset sang SSD cục bộ của Colab (/content/dataset)
# - True: Copy/Giải nén sang SSD Colab (Nạp video cực nhanh, tránh nghẽn I/O Drive)
# - False: Đọc trực tiếp từ Google Drive
COPY_TO_LOCAL_SSD = True

# Các đường dẫn dữ liệu
DRIVE_DATASET_DIR = os.path.join(DRIVE_PROJECT_ROOT, "dataset")
DRIVE_DATASET_ZIP = os.path.join(DRIVE_PROJECT_ROOT, "dataset.zip")
LOCAL_DATASET_DIR = "/content/dataset"

# 2. Xử lý dữ liệu theo tùy chọn
if COPY_TO_LOCAL_SSD:
    print("[*] Đang chuẩn bị dữ liệu trên SSD Colab (/content/dataset)...")
    if os.path.exists(os.path.join(LOCAL_DATASET_DIR, "captions")):
        print("[+] Thư mục dataset cục bộ đã sẵn sàng, bỏ qua bước copy.")
    elif os.path.exists(DRIVE_DATASET_ZIP):
        print(f"[*] Tìm thấy file nén {DRIVE_DATASET_ZIP}. Đang giải nén vào /content...")
        with zipfile.ZipFile(DRIVE_DATASET_ZIP, 'r') as zip_ref:
            zip_ref.extractall("/content")
        print("[+] Giải nén hoàn tất vào SSD Colab!")
    elif os.path.exists(DRIVE_DATASET_DIR):
        print(f"[*] Đang sao chép từ Drive ({DRIVE_DATASET_DIR}) sang SSD Colab ({LOCAL_DATASET_DIR})...")
        shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
        print("[+] Sao chép hoàn tất vào SSD Colab!")
    else:
        print(f"[!] CẢNH BÁO: Không tìm thấy {DRIVE_DATASET_DIR} trên Drive. Sử dụng trực tiếp từ Drive.")
        LOCAL_DATASET_DIR = DRIVE_DATASET_DIR
    DATA_ROOT = LOCAL_DATASET_DIR
else:
    print("[*] Chế độ: Đọc dữ liệu trực tiếp từ Google Drive.")
    DATA_ROOT = DRIVE_DATASET_DIR

# 3. Đưa thư mục src vào sys.path để import trực tiếp các module
import sys
SRC_DIR = os.path.join(DRIVE_PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"\n[+] DATA_ROOT hiện tại: {DATA_ROOT}")
print(f"[+] SRC_DIR hiện tại: {SRC_DIR}")


## 🏋️ Cell 2: Cấu Hình & Chạy Huấn Luyện DINOv2-TVPR

- Nạp cấu hình tối ưu từ `default.yaml` với kiến trúc DINOv2 Foundation.
- Tự động liên kết `data_root` vào SSD Colab để đọc dữ liệu siêu tốc.
- Trỏ đường dẫn lưu checkpoint (`save_dir`) **trực tiếp về Google Drive**, đảm bảo mô hình (`best.pth`, `latest.pth`, `train_history.json`) **không bao giờ bị mất** khi Colab ngắt kết nối.


In [ ]:
# ==============================================================================
# CELL 2: CẤU HÌNH & CHẠY HUẤN LUYỆN DINOV2-TVPR
# ==============================================================================
import yaml
import os

# 1. Nạp cấu hình gốc và tinh chỉnh cho Colab
base_config_path = os.path.join(SRC_DIR, "configs", "default.yaml")
with open(base_config_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# Cập nhật đường dẫn cho Colab
cfg["data"]["data_root"] = DATA_ROOT
# Checkpoints lưu trực tiếp về Google Drive để bảo toàn dữ liệu khi ngắt phiên
cfg["train"]["save_dir"] = os.path.join(DRIVE_PROJECT_ROOT, "src", "checkpoints")

# Đảm bảo chạy kiến trúc DINOv2
cfg["model"]["arch"] = "dinov2"

# --- TÙY CHỌN TIẾP TỤC HUẤN LUYỆN (RESUME TRAINING) ---
# - None: Huấn luyện mới từ đầu
# - "latest": Tự động nạp latest.pth trên Google Drive để huấn luyện tiếp
# - "best": Tự động nạp best.pth để tiếp tục cải thiện từ mốc tốt nhất
RESUME_CHOICE = None  # Đổi thành "latest" hoặc "best" nếu muốn train tiếp
if RESUME_CHOICE:
    cfg["train"]["resume"] = RESUME_CHOICE

colab_config_path = "/content/colab_config.yaml"
with open(colab_config_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True)

print(f"[+] Đã tạo file cấu hình Colab: {colab_config_path}")
print(f"    - Kiến trúc: {cfg['model'].get('arch', 'dinov2')}")
print(f"    - Vision Backbone: {cfg['model'].get('dinov2_model_name', 'facebook/dinov2-small')}")
print(f"    - Text Backbone: {cfg['model'].get('bert_model_name', 'vinai/phobert-base-v2')}")
print(f"    - Dữ liệu (data_root): {cfg['data']['data_root']}")
print(f"    - Nơi lưu Checkpoint (Drive): {cfg['train']['save_dir']}")
print(f"    - Tiếp tục huấn luyện (Resume): {cfg['train'].get('resume', 'Không (train mới)')}")
print(f"    - Batch size: {cfg['data'].get('batch_size', 32)}")
print(f"    - Số epochs: {cfg['train']['epochs']}")

# 2. Chuyển thư mục làm việc sang src và bắt đầu huấn luyện
%cd {SRC_DIR}

print("\n[*] Bắt đầu quá trình huấn luyện DINOv2-TVPR...")
!python train.py --config /content/colab_config.yaml


## 🌐 Cell 3: Web App Tương Tác Gradio (Chạy Trực Tiếp Trong Cell)

- **Chạy hoàn toàn bằng Python thuần** trực tiếp trong cell này (không gọi qua lệnh `!python`).
- Tích hợp `MFGFTrackingEngine`: Tự động nhận diện mô hình DINOv2 hoặc MFGF, tải video hoặc chọn video mẫu từ dataset, nhập câu mô tả tiếng Việt (Ví dụ: *"Người phụ nữ mặc áo khoác trắng và quần xanh dương"*).
- Hệ thống phát hiện người liên tục, đối soát đặc trưng DINOv2 + PhoBERT, vẽ bounding box neon và xuất video tracking mượt mà.
- Tự động sinh đường link công khai (`https://xxxx.gradio.live`) với `share=True` để trải nghiệm demo trên điện thoại hoặc chia sẻ từ xa!


In [ ]:
# ==============================================================================
# CELL 3: WEB APP TƯƠNG TÁC GRADIO (CHẠY TRỰC TIẾP TRONG CELL NÀY)
# ==============================================================================
import os
import sys
import cv2
import gradio as gr
import torch

# 1. Đảm bảo đường dẫn mã nguồn src nằm trong sys.path
if 'SRC_DIR' in locals() and SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
elif '/content/drive/MyDrive/PBL6/src' not in sys.path:
    sys.path.insert(0, '/content/drive/MyDrive/PBL6/src')

from pipeline_tracking import MFGFTrackingEngine

# 2. Quản lý Engine Tracking (Lazy Loading)
ckpt_dir = os.path.join(DRIVE_PROJECT_ROOT, "src", "checkpoints") if 'DRIVE_PROJECT_ROOT' in locals() else "/content/drive/MyDrive/PBL6/src/checkpoints"
best_ckpt = os.path.join(ckpt_dir, "best.pth")
latest_ckpt = os.path.join(ckpt_dir, "latest.pth")

engine_cache = {}

def get_engine(ckpt_path: str):
    if ckpt_path not in engine_cache:
        print(f"[*] Đang khởi tạo TrackingEngine với checkpoint: {ckpt_path}", flush=True)
        colab_cfg = "/content/colab_config.yaml" if os.path.exists("/content/colab_config.yaml") else None
        engine = MFGFTrackingEngine(config_path=colab_cfg, checkpoint_path=ckpt_path)
        engine_cache[ckpt_path] = engine
    return engine_cache[ckpt_path]

def run_tvpr_tracking(video_input, text_query, ckpt_choice, top_k, progress=gr.Progress()):
    if not video_input:
        return None, "Vui lòng tải lên hoặc chọn một video!"
    if not text_query or not text_query.strip():
        return None, "Vui lòng nhập câu mô tả người bằng tiếng Việt!"

    selected_ckpt = best_ckpt if ckpt_choice == "best.pth" else latest_ckpt
    if not os.path.exists(selected_ckpt):
        if os.path.exists(latest_ckpt):
            selected_ckpt = latest_ckpt
        elif os.path.exists(best_ckpt):
            selected_ckpt = best_ckpt
        else:
            selected_ckpt = None

    try:
        progress(0.05, desc="Đang chuẩn bị mô hình...")
        engine = get_engine(selected_ckpt)

        output_dir = "/content/outputs"
        os.makedirs(output_dir, exist_ok=True)
        raw_output_path = os.path.join(output_dir, f"tracking_{os.path.basename(video_input)}")

        def progress_cb(pct, msg):
            progress(pct, desc=msg)

        progress(0.15, desc="Đang quét người và đối soát ngữ nghĩa...")
        result = engine.process_and_track(
            video_path=video_input,
            query_text=text_query,
            output_path=raw_output_path,
            top_k=int(top_k),
            progress_callback=progress_cb
        )

        result_path = result["output_path"]

        # Chuyển đổi mã hóa sang H.264 để trình duyệt web phát trực tiếp mượt mà
        progress(0.9, desc="Đang tối ưu định dạng video H.264 cho trình duyệt...")
        web_video_path = result_path.replace(".mp4", "_web.mp4")
        os.system(f"ffmpeg -y -i \"{result_path}\" -vcodec libx264 -crf 23 -pix_fmt yuv420p \"{web_video_path}\" >/dev/null 2>&1")
        final_video_path = web_video_path if os.path.exists(web_video_path) else result_path

        info_text = (
            f"=== KẾT QUẢ TRACKING TVPR ===\n"
            f"• Trạng thái: Hoàn tất thành công!\n"
            f"• Video: {os.path.basename(video_input)}\n"
            f"• Mô tả truy vấn: {text_query}\n"
            f"• Kiến trúc mô hình: {getattr(engine, 'arch', 'DINOv2')}\n"
            f"• Checkpoint: {os.path.basename(selected_ckpt) if selected_ckpt else 'Khởi tạo sẵn'}\n"
            f"• Mục tiêu chính: ID #{result.get('target_id', 1)}\n"
            f"• Độ tin cậy (Confidence): {result.get('display_score', 0.0):.1f}%\n"
            f"• Khung hình xuất hiện: Frame {result.get('start_frame', 0)} -> {result.get('exit_frame', 0)}\n"
            f"• Rời khỏi góc máy (Out-of-Camera): {'Có' if result.get('out_of_camera', False) else 'Không'}\n"
            f"• Tổng số khung hình đã xử lý: {result.get('total_frames', 0)}"
        )
        progress(1.0, desc="Hoàn tất!")
        return final_video_path, info_text

    except Exception as e:
        import traceback
        traceback.print_exc()
        return None, f"Đã xảy ra lỗi trong quá trình xử lý: {str(e)}"

# 3. Tìm danh sách video mẫu nếu có trong dataset
sample_examples = []
videos_dir = os.path.join(DATA_ROOT, "videos") if 'DATA_ROOT' in locals() and os.path.exists(os.path.join(DATA_ROOT, "videos")) else ""
if videos_dir and os.path.exists(videos_dir):
    for fname in sorted(os.listdir(videos_dir))[:4]:
        if fname.endswith(('.mp4', '.avi')):
            sample_examples.append([os.path.join(videos_dir, fname), "Người phụ nữ mặc áo khoác trắng và quần xanh dương."])

# 4. Thiết kế giao diện Gradio
with gr.Blocks(title="DINOv2 TVPR - Colab Demo", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎥 DINOv2-TVPR: Tìm Kiếm & Tracking Người Qua Video Bằng Tiếng Việt")
    gr.Markdown(
        "Mô hình Foundation **Meta DINOv2** kết hợp **VinAI PhoBERT v2** và **Temporal Attention Aggregator** "
        "cho bài toán Text-to-Video Person Retrieval & Continuous Tracking."
    )

    with gr.Row():
        with gr.Column(scale=5):
            video_input = gr.Video(label="1. Video Đầu Vào (Upload hoặc chọn mẫu bên dưới)", sources=["upload"])
            text_query = gr.Textbox(
                label="2. Câu Mô Tả Người Cần Tìm (Tiếng Việt)",
                value="Người phụ nữ mặc áo khoác trắng và quần xanh dương.",
                placeholder="Nhập chi tiết ngoại hình, màu sắc quần áo, phụ kiện..."
            )
            with gr.Row():
                ckpt_choice = gr.Dropdown(
                    choices=["best.pth", "latest.pth"],
                    value="best.pth",
                    label="3. Chọn Checkpoint Huấn Luyện"
                )
                top_k = gr.Slider(minimum=1, maximum=5, value=1, step=1, label="4. Số Lượng Mục Tiêu (Top-K)")
            
            submit_btn = gr.Button("🚀 Bắt Đầu Tìm Kiếm & Theo Vết", variant="primary", size="lg")

            if sample_examples:
                gr.Examples(
                    examples=sample_examples,
                    inputs=[video_input, text_query],
                    label="Ví Dụ Mẫu Từ Dataset"
                )

        with gr.Column(scale=6):
            video_output = gr.Video(label="Video Kết Quả (Khung Theo Vết Neon & Độ Tin Cậy)")
            info_output = gr.Textbox(label="Thông Tin Chi Tiết Tracking", lines=10)

    submit_btn.click(
        fn=run_tvpr_tracking,
        inputs=[video_input, text_query, ckpt_choice, top_k],
        outputs=[video_output, info_output]
    )

# 5. Khởi chạy trực tiếp trong Notebook và tạo đường link công khai (Public URL)
print("[*] Đang khởi chạy Gradio Web App...")
demo.launch(share=True, debug=True, inline=True)
